In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math
import numpy as np
from framework import rss, rss_rel
from framework import Measurement as m

In [33]:
ghg = [14,11.87,14.52,13,20.5,21] # data from https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2023.1147016/full
print(f"\nMean GHG emissions: {np.mean(ghg)} gCO2eq/kWh")
print(f"Standard Deviation: {np.std(ghg)} gCO2eq/kWh")
ghg_large = m(np.mean(ghg), np.std(ghg))
ghg_smr = ghg_large * 1.09
print(f"\nLarge Reactor GHG emissions: {ghg_large} gCO2eq/kWh")
print(f"SMR GHG emissions: {ghg_smr} gCO2eq/kWh")

countries = ["IT", "PO", "CH", "FR"]
reactors = ["APR-1400", "EPR-1750", "AP-1000", "BWRX-300", "Rolls Royce SMR", "NuScale SMR"]
reactors_ghgei = [ghg_large, ghg_large, ghg_large, ghg_smr, ghg_smr, ghg_smr]
reactors_powers = [1340, 1630, 1120, 300, 470, 77]  # in MW
# Generate a gaussian distribution centered at 0.9, clipped between 0.85 and 0.95
np.random.seed(0)
Capacity_Factor = np.clip(np.random.normal(loc=0.9, scale=0.02, size=100), 0.85, 0.95)

hours_year = 8760

GHGei_countries = [m(238.50, 64.67), m(703.17, 129.29), m(39.09, 22.13), m(332.55, 119.03)] # [gCO2eq/kWh] = [KgCO2eq/MWh]


Mean GHG emissions: 15.815 gCO2eq/kWh
Standard Deviation: 3.589149156369329 gCO2eq/kWh

Large Reactor GHG emissions: 15.815 ± 3.58915 gCO2eq/kWh
SMR GHG emissions: 17.2384 ± 3.91217 gCO2eq/kWh


In [34]:
for i, country in enumerate(countries):
    print("\n")
    df = pd.read_csv(f"{country}.csv")

    # Read ../Total Loads/country.csv to get total load
    load_df = pd.read_csv(f"../Total Loads/{country}.csv")
    # Get the last column (total load)
    total_load = load_df.iloc[:, -1]
    total_load_sum = total_load.sum()/1e3 # Convert to GWh/year
    display(Math(rf"\text{{{country}}} \text{{ Total Load}}: {total_load_sum:.2f} \ \mathrm{{GWh/year}}"))

    # # Mean +- Standard Deviation
    # carbon_intensity_lifecycle = df["Carbon intensity gCO₂eq/kWh (Life cycle)"].mean()
    # carbon_intensity_lifecycle_std = df["Carbon intensity gCO₂eq/kWh (Life cycle)"].std()
    # GHGei_today = m(carbon_intensity_lifecycle, carbon_intensity_lifecycle_std)

    # Median (Lower - Upper) - Using Interquartile Range (IQR)
    carbon_intensity_lifecycle_median = df["Carbon intensity gCO₂eq/kWh (Life cycle)"].median()
    percentage_of_renewables = df["Renewable energy percentage (RE%)"].mean()
    percentage_of_lowcarbon = df["Carbon-free energy percentage (CFE%)"].mean()
    display(Math(rf"\text{{{country}}} \text{{ Grid Mix}}: {percentage_of_renewables:.2f}\% \ \text{{Renewables}}, \ {percentage_of_lowcarbon:.2f}\% \ \text{{Low-Carbon}}"))
    carbon_intensity_lifecycle_IQR = df['Carbon intensity gCO₂eq/kWh (Life cycle)'].quantile(0.75) - df['Carbon intensity gCO₂eq/kWh (Life cycle)'].quantile(0.25)
    GHGei_today = m(carbon_intensity_lifecycle_median, carbon_intensity_lifecycle_IQR/2)

    display(Math(rf"\text{{{country}}} \text{{ GHG Emissions}}: {GHGei_today} \ \mathrm{{gCO_2eq/kWh}}"))
    
    for i in range(len(reactors)):

        # Estimate Energy produced by a reactor in a year
        Energy_reactor = []
        for cf in Capacity_Factor:
            Energy_reactor.append(reactors_powers[i] * cf * hours_year / 1000) # Converted to GWh
        Energy_reactor = m(np.mean(Energy_reactor), np.std(Energy_reactor)) # GWh/year
        
        # Total GHG emissions in a year as of today
        GHGe_pre = GHGei_today * total_load_sum  # gCO2/kWh = KgCO2/MWh = tonCO2/GWh * GWh/year
        # Total GHG emissions in a year if 1 unit of the reactor was to be build right now
        GHGe_post = (reactors_ghgei[i] * Energy_reactor + GHGei_today * (total_load_sum - Energy_reactor) )
        # display(f"{reactors[i]}: {reactors_ghgei[i] * Energy_reactor /1e3}")
        # display(f"{GHGei_today * total_load_sum / 1e3}")
        # Reduction in GHG emissions
        # reduction = rss_rel(lambda ghge_pre, ghge_post: (ghge_pre - ghge_post)/ghge_pre, GHGe_pre, GHGe_post)
        difference = rss_rel(lambda ghge_pre, ghge_post: (ghge_pre - ghge_post), GHGe_pre, GHGe_post)
        display(Math(rf"\text{{Current GHG emissions in {country}: }} {GHGe_pre / 1e6} \ \mathrm{{MtonsCO_2eq/year}}"))
        display(Math(rf"\text{{GHG emissions after installing 1 {reactors[i]} reactor in {country}: }} {GHGe_post / 1e6} \ \mathrm{{MtonsCO_2eq/year}}"))
        # difference = GHGe_pre - GHGe_post

        

        # display(Math(rf"\text{{By installing 1 {reactors[i]} reactor in {country}, the GHG emissions would be reduced by }} {reduction*100} \%"))
        display(Math(rf"\text{{By installing 1 {reactors[i]} reactor in {country}, the GHG emissions would be reduced by }} {difference/1e6} \ \mathrm{{MtonsCO_2eq/year}}"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>